# 04 Geometry Shortcut Diagnosis

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`exp010_geometry_distribution.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp010_geometry_distribution.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)
- [`exp011_012_causal_probes.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp011_012_causal_probes.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)
- [`exp013_geometry_shortcut_binary.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp013_geometry_shortcut_binary.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)
- [`exp010_012_aggregate.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp010_012_aggregate.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)
- [`exp013_014_aggregate.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp013_014_aggregate.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/MLP/Normal/scripts/exp010_geometry_distribution.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'd8753183b5a80ab425d05394850277492aa8b074'}, '05_model_optimize/MLP/Normal/scripts/exp011_012_causal_probes.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'f60144ca0a2264195259d5760668a033ee556ec3'}, '05_model_optimize/MLP/Normal/scripts/exp013_geometry_shortcut_binary.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '8808787653693e0b9cedeed19c6360cee5a86ef9'}, '05_model_optimize/MLP/Normal/scripts/exp010_012_aggregate.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '4157bd9d506bbae337a260d3c98191d38c27ee98'}, '05_model_optimize/MLP/Normal/scripts/exp013_014_aggregate.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '5fa22a60cd93a8626323f1228395c4095a47129b'}}

## exp010_geometry_distribution

원본: [05_model_optimize/MLP/Normal/scripts/exp010_geometry_distribution.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp010_geometry_distribution.py)  
Git blob: `d8753183b5a80ab425d05394850277492aa8b074`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

EXP유010 — Official Training vs Hand-made geometry, Original/Size/Size->CoM.

Reuses EXACT EXP유006 apply_size and center_batch; all source data are developer
train/validation/handmade only. No MLP fitting or test access.

CLI 인자 (정적 추출): `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Normal/scripts/exp010_geometry_distribution.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""EXP유010 — Official Training vs Hand-made geometry, Original/Size/Size->CoM.

Reuses EXACT EXP유006 apply_size and center_batch; all source data are developer
train/validation/handmade only. No MLP fitting or test access.
"""
from __future__ import annotations
import argparse
import csv
import json
from pathlib import Path
import numpy as np
from scipy.stats import wasserstein_distance

from exp006_size_com_comparison import (
    ROOT, WRITERS, LABELS, load_official, load_writers,
    image_geometry, process, save_csv
)
from Automatic_Centering_Transformer import center_shifts

FEATURES=("width","height","long_side","cx","cy","com_offset",
          "bbox_cx","bbox_cy","ink_ratio","mass","edge_mass")
METHODS=("raw","size","size_com")

def get_processed(raw,method,target):
    if method=="raw":
        return np.asarray(raw,dtype=np.float32).reshape(-1,784)
    return process(raw,method,target)

def extract(raw,method,target,batch_size=1024):
    n=len(raw)
    full={k:np.empty(n,dtype=np.float64) for k in FEATURES}
    full["scale"]=np.ones(n,dtype=np.float64)
    full["shift_x"]=np.zeros(n,dtype=np.float64)
    full["shift_y"]=np.zeros(n,dtype=np.float64)
    for start in range(0,n,batch_size):
        end=min(n,start+batch_size)
        v=np.asarray(raw[start:end],dtype=np.float32).reshape(-1,784)
        if method=="size_com":
            presize=process(v,"size",target)
            shift=center_shifts(presize,"mass")
            full["shift_x"][start:end]=shift[:,0]
            full["shift_y"][start:end]=shift[:,1]
            processed=process(v,"size_com",target)
        elif method=="size":
            processed=process(v,"size",target)
        else:
            processed=v
        before=image_geometry(v)
        geo=image_geometry(processed)
        if method!="raw":
            full["scale"][start:end]=float(target)/before["long_side"]
        for key in FEATURES:
            full[key][start:end]=geo[key]
        if start%16384==0:
            print(f"GEOMETRY {method} {start}:{end}/{n}",flush=True)
    return full

def summarize(dataset,method,labels,writers,g):
    groups={"all":np.ones(len(labels),dtype=bool),
            "digits":labels<10,"letters":labels>=10}
    for i,cl in enumerate(LABELS):
        groups["class_"+cl]=labels==i
    if writers is not None:
        for w in WRITERS:
            groups["writer_"+str(w)]=writers==w
            groups["writer_"+str(w)+"_digits"]=(writers==w)&(labels<10)
            groups["writer_"+str(w)+"_letters"]=(writers==w)&(labels>=10)
    result=[]
    for group,mask in groups.items():
        if not mask.any():continue
        for k in (*FEATURES,"scale","shift_x","shift_y"):
            x=g[k][mask]
            result.append({"dataset":dataset,"preprocessing":method,"group":group,
                "feature":k,"n":len(x),"mean":float(x.mean()),
                "median":float(np.median(x)),
                "std":float(x.std()),
                "p10":float(np.quantile(x,.1)),
                "p90":float(np.quantile(x,.9))})
    return result

def run(out):
    out=Path(out)
    if out.exists():raise FileExistsError(out)
    out.mkdir(parents=True)
    official=load_official(ROOT/"02_data"/"datasets"/"MNIST_EMNIST_official_split_112000.npz")
    handmade=load_writers(ROOT/"02_data"/"Hand_made")
    assert official["X_train"].shape==(76800,784)
    assert official["X_val"].shape==(19200,784)
    target=int(np.median(image_geometry(official["X_train"])["long_side"]))
    assert target==20,"EXP006 target mismatch"
    x_hm=np.concatenate([handmade[w]["X"] for w in WRITERS],axis=0)
    y_hm=np.concatenate([handmade[w]["y"] for w in WRITERS],axis=0)
    writer_ids=np.repeat(WRITERS,640)
    sources={
      "OfficialTrain":(official["X_train"],official["y_train"],None),
      "OfficialVal":(official["X_val"],official["y_val"],None),
      "Handmade":(x_hm,y_hm,writer_ids)
    }
    all_geo={}
    summary=[]
    per_handmade=[]
    for dataset,(X,y,wids) in sources.items():
        for method in METHODS:
            g=extract(X,method,target)
            all_geo[dataset,method]=g
            summary+=summarize(dataset,method,y,wids,g)
            if dataset=="Handmade":
                for i in range(len(X)):
                    per_handmade.append({
                        "preprocessing":method,
                        "writer":int(wids[i]),
                        "index":int(i%640),
                        "label":LABELS[int(y[i])],
                        **{k:float(g[k][i]) for k in (*FEATURES,"scale","shift_x","shift_y")}
                    })
    save_csv(out/"geometry_summary.csv",summary)
    save_csv(out/"handmade_geometry_per_image.csv",per_handmade)
    distances=[]
    for method in METHODS:
        for class_id,cl in enumerate(LABELS):
            for segment,mask_train,mask_hm,mask_val in [
                ("all",np.ones(76800,dtype=bool),
                 np.ones(2560,dtype=bool),np.ones(19200,dtype=bool)),
                ("class_"+cl,official["y_train"]==class_id,
                 y_hm==class_id,official["y_val"]==class_id)
            ]:
                # Evaluate all once; individual classes 16x, all only once
                if segment=="all" and class_id>0:continue
                train=all_geo["OfficialTrain",method]
                val=all_geo["OfficialVal",method]
                hm=all_geo["Handmade",method]
                for feature in ("width","height","long_side","cx","cy",
                                "com_offset","ink_ratio","mass"):
                    distances.append({"preprocessing":method,"segment":segment,
                        "feature":feature,
                        "n_train":int(mask_train.sum()),
                        "n_val":int(mask_val.sum()),
                        "n_handmade":int(mask_hm.sum()),
                        "train_vs_handmade_w1":float(wasserstein_distance(
                            train[feature][mask_train],hm[feature][mask_hm])),
                        "train_vs_val_w1":float(wasserstein_distance(
                            train[feature][mask_train],val[feature][mask_val]))})
    save_csv(out/"geometry_distance.csv",distances)
    manifest={"experiment":"Normal-EXP010","preprocessing_methods":list(METHODS),
      "code":"original exp006 image_geometry, process + center_shifts",
      "target_bbox_longside":target,"foreground_threshold":32,
      "official_train_n":76800,"official_validation_n":19200,
      "handmade_n":2560,"writers":list(WRITERS),"labels":LABELS,
      "official_sha256":official["sha256"],
      "writer_sha256":{str(w):handmade[w]["sha256"] for w in WRITERS},
      "official_original_test_used":False,"professor_test_used":False,
      "handmade_training_used":False,"model_training_used":False}
    (out/"manifest.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8")
    print("EXP010_GEOMETRY_COMPLETE",flush=True)
    for cl in ("B","D","8","0"):
        for method in METHODS:
            g_train=all_geo["OfficialTrain",method]
            g_hm=all_geo["Handmade",method]
            clidx=LABELS.index(cl)
            print("EXAMPLE",cl,method,
                  "train_long_p50",np.median(g_train["long_side"][official["y_train"]==clidx]),
                  "handmade_long_p50",np.median(g_hm["long_side"][y_hm==clidx]),
                  flush=True)

if __name__=="__main__":
    p=argparse.ArgumentParser()
    p.add_argument("--out",required=True,type=Path)
    a=p.parse_args()
    run(a.out)


## exp011_012_causal_probes

원본: [05_model_optimize/MLP/Normal/scripts/exp011_012_causal_probes.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp011_012_causal_probes.py)  
Git blob: `f60144ca0a2264195259d5760668a033ee556ec3`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

EXP유011 & EXP유012: original EXP006 model fixed and matched inputs.

Use archived exact 9 EXP006 models from successful GitHub Actions run 43785390739
(C.O.M., Size-only, Size->CoM, each seed42-44). NO retraining. The original
model's diagonal predictions must exactly match archived manifests and per-group
correct counts before counterfactual probes are accepted.

CLI 인자 (정적 추출): `--seed`, `--models-root`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Normal/scripts/exp011_012_causal_probes.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""EXP유011 & EXP유012: original EXP006 model fixed and matched inputs.

Use archived exact 9 EXP006 models from successful GitHub Actions run 43785390739
(C.O.M., Size-only, Size->CoM, each seed42-44). NO retraining. The original
model's diagonal predictions must exactly match archived manifests and per-group
correct counts before counterfactual probes are accepted.
"""
from __future__ import annotations
import argparse
import csv
import json
from pathlib import Path

import joblib
import numpy as np
from scipy.ndimage import affine_transform

from exp006_size_com_comparison import (
    ROOT, WRITERS, LABELS, PAIRS, load_official, load_writers,
    image_geometry, process, save_csv, sha256
)
from Automatic_Centering_Transformer import center_batch,center_shifts
from stroke_normalization import topology

SEEDS=(42,43,44)
TRAINED=("com","size","size_com")
EVALS=("raw","com","size","size_com")
SCALES=(.8,.9,1.0,1.1,1.2)
COMPETITOR={"B":"8","8":"B","D":"0","0":"D","F":"5","5":"F"}

def evaluate_groups(truth,pred,writers=None):
    groups={"all":np.ones(len(truth),bool),
            "digits":truth<10,"letters":truth>=10}
    for i,cl in enumerate(LABELS):
        groups["class_"+cl]=truth==i
    if writers is not None:
        for w in WRITERS:
            groups[f"writer_{w}"]=writers==w
            groups[f"writer_{w}_digits"]=(writers==w)&(truth<10)
            groups[f"writer_{w}_letters"]=(writers==w)&(truth>=10)
    out={}
    for name,mask in groups.items():
        n=int(mask.sum())
        if n:
            out[name]={"n":n,"correct":int(np.sum(pred[mask]==truth[mask])),
                       "accuracy":float(np.mean(pred[mask]==truth[mask]))}
    return out

def fixed_scale(batch,scale):
    """Uniform bbox-centered affine, identical order/cval as original EXP006."""
    a=np.asarray(batch,dtype=np.float32).reshape(-1,28,28)
    if scale==1.0:return a.copy().reshape(-1,784)
    g=image_geometry(a)
    out=np.empty_like(a)
    inv=1.0/scale
    for i,im in enumerate(a):
        cy,cx=g["bbox_cy"][i],g["bbox_cx"][i]
        out[i]=np.clip(affine_transform(im,np.eye(2)*inv,
                    offset=(cy*(1-inv),cx*(1-inv)),output_shape=(28,28),
                    order=1,mode="constant",cval=0.0,prefilter=False),0,255)
    return out.reshape(-1,784)

def paired(a,b,mask):
    old=a[mask];new=b[mask]
    return {"n":int(mask.sum()),
      "old_correct":int(old.sum()),"new_correct":int(new.sum()),
      "fixed":int((~old&new).sum()),
      "new_wrong":int((old&~new).sum()),
      "net_correct":int(new.sum()-old.sum()),
      "delta_accuracy_pp":float(100*(new.mean()-old.mean()))}

def run(seed,models_root,out):
    out=Path(out)
    if out.exists():raise FileExistsError(out)
    out.mkdir(parents=True)
    official=load_official(ROOT/"02_data"/"datasets"/"MNIST_EMNIST_official_split_112000.npz")
    hm=load_writers(ROOT/"02_data"/"Hand_made")
    raw_handmade=np.concatenate([hm[w]["X"] for w in WRITERS])
    y_hm=np.concatenate([hm[w]["y"] for w in WRITERS]).astype(int)
    writer_ids=np.repeat(WRITERS,640)
    y_val=np.asarray(official["y_val"],dtype=int)
    assert len(raw_handmade)==2560 and official["X_val"].shape==(19200,784)
    target=int(np.median(image_geometry(official["X_train"])["long_side"]))
    assert target==20
    raw_sets={"OfficialVal":official["X_val"],"Handmade":raw_handmade}
    preprocessed={}
    for dataset,raw in raw_sets.items():
        for method in EVALS:
            preprocessed[dataset,method]=(
              np.asarray(raw,dtype=np.float32).reshape(-1,784) if method=="raw"
              else process(raw,method,target)
            )
    models={}
    historical={}
    for method in TRAINED:
        folder=Path(models_root)/f"normal-exp006-{method}-{seed}"
        manifest=json.loads((folder/"run_manifest.json").read_text(encoding="utf-8"))
        assert manifest["method"]==method and manifest["seed"]==seed
        assert manifest["official_sha256"]==official["sha256"]
        assert all(manifest["writer_sha256"][str(w)]==hm[w]["sha256"] for w in WRITERS)
        assert not manifest["official_test_used"] and not manifest["professor_test_used"]
        assert not manifest["handmade_used_in_fit"]
        model_path=folder/"model.joblib"
        assert sha256(model_path)==manifest["model_sha256"],"Archived checkpoint hash mismatch"
        models[method]=joblib.load(model_path)
        historical[method]=manifest
        print("LOADED AND HASH VERIFIED",method,seed,flush=True)

    cross_metrics=[]
    cross_predictions=[]
    prediction_lookup={}
    for method,model in models.items():
        for dataset in ("OfficialVal","Handmade"):
            truth=y_val if dataset=="OfficialVal" else y_hm
            writers=None if dataset=="OfficialVal" else writer_ids
            for representation in EVALS:
                X=preprocessed[dataset,representation]
                pred=np.asarray(model.predict(X),dtype=int)
                assert len(pred)==len(truth)
                if representation==method:
                    g=evaluate_groups(truth,pred,writers)
                    for group in ("all","digits","letters"):
                        historical_correct=historical[method][
                            "official_validation" if dataset=="OfficialVal" else "handmade"
                        ][group]["correct"]
                        actual=g[group]["correct"]
                        if actual!=historical_correct:
                            raise AssertionError(f"HISTORICAL MISMATCH {method} {seed} "
                              f"{dataset} {group}: recorded={historical_correct} current={actual}")
                    print("EXACT ORIGINAL MODEL/RESULT MATCH",method,seed,dataset,
                          "correct",g["all"]["correct"],flush=True)
                group_data=evaluate_groups(truth,pred,writers)
                for group,val in group_data.items():
                    cross_metrics.append({"seed":seed,"trained":method,
                        "dataset":dataset,"input":representation,"group":group,**val})
                if dataset=="Handmade":
                    prediction_lookup[method,representation]=pred
                    for i,(t,p) in enumerate(zip(truth,pred)):
                        cross_predictions.append({"seed":seed,"trained":method,
                            "input":representation,"writer":int(writer_ids[i]),
                            "index":int(i%640),"true":LABELS[t],
                            "pred":LABELS[p],"correct":int(t==p)})
                print("EXP011_CROSS",seed,method,dataset,representation,
                      "all",f"{group_data['all']['accuracy']*100:.3f}%",
                      "digits",f"{group_data['digits']['accuracy']*100:.3f}%",
                      "letters",f"{group_data['letters']['accuracy']*100:.3f}%",flush=True)
    save_csv(out/"cross_metrics.csv",cross_metrics)
    save_csv(out/"cross_predictions.csv",cross_predictions)
    paired_results=[]
    diag_old=(prediction_lookup["size","size"]==y_hm)
    diag_new=(prediction_lookup["size_com","size_com"]==y_hm)
    for group,mask in (("all",np.ones(len(y_hm),bool)),
                       ("digits",y_hm<10),("letters",y_hm>=10),
                       *[(f"class_{cl}",y_hm==j) for j,cl in enumerate(LABELS)]):
        result=paired(diag_old,diag_new,mask)
        paired_results.append({"seed":seed,"contrast":"size diagonal -> size_com diagonal",
                               "group":group,**result})
        for method in TRAINED:
            a=(prediction_lookup[method,"size"]==y_hm)
            b=(prediction_lookup[method,"size_com"]==y_hm)
            paired_results.append({"seed":seed,"contrast":f"fixed model={method}: size input -> size_com input",
                                   "group":group,**paired(a,b,mask)})
        for rep in ("size","size_com"):
            a=(prediction_lookup["size",rep]==y_hm)
            b=(prediction_lookup["size_com",rep]==y_hm)
            paired_results.append({"seed":seed,"contrast":f"fixed input={rep}: size train -> size_com train",
                                   "group":group,**paired(a,b,mask)})
    save_csv(out/"paired_effects.csv",paired_results)
    pair_errors=[]
    for method in TRAINED:
        for inp in EVALS:
            p=prediction_lookup[method,inp]
            for a,b in PAIRS:
                pair_errors.append({"seed":seed,"trained":method,"input":inp,
                   "true":a,"pred":b,
                   "n_errors":int(np.sum((y_hm==LABELS.index(a)) &
                                             (p==LABELS.index(b))))})
    save_csv(out/"pair_errors.csv",pair_errors)

    # EXP011 controls: geometry before/after CoM on the same Size-only image.
    sized=preprocessed["Handmade","size"]
    corrected=preprocessed["Handmade","size_com"]
    sized_geo=image_geometry(sized)
    centered_geo=image_geometry(corrected)
    shifts=center_shifts(sized,"mass")
    geometry=[]
    for i in range(len(y_hm)):
        geometry.append({"seed":seed,"writer":int(writer_ids[i]),
            "index":int(i%640),"class":LABELS[int(y_hm[i])],
            "size_com_shift_x":int(shifts[i,0]),"size_com_shift_y":int(shifts[i,1]),
            "size_com_shift_distance":float(np.hypot(*shifts[i])),
            "com_distance_before":float(sized_geo["com_offset"][i]),
            "com_distance_after":float(centered_geo["com_offset"][i]),
            "long_side_before":int(sized_geo["long_side"][i]),
            "long_side_after":int(centered_geo["long_side"][i]),
            "mass_before":float(sized_geo["mass"][i]),
            "mass_after":float(centered_geo["mass"][i]),
            "edge_mass_before":float(sized_geo["edge_mass"][i]),
            "edge_mass_after":float(centered_geo["edge_mass"][i]),
            "size_model_size_correct":int(diag_old[i]),
            "size_com_model_size_com_correct":int(diag_new[i]),
            "same_size_model_after_com_correct":int(
                 prediction_lookup["size","size_com"][i]==y_hm[i]),
            "same_size_com_model_before_com_correct":int(
                 prediction_lookup["size_com","size"][i]==y_hm[i]),
            })
    save_csv(out/"size_com_geometry_and_outcomes.csv",geometry)

    # EXP012: perturb SAME image's uniform BBox size while re-centering each
    # perturbed image, keeping source class and model fixed.
    raw=np.asarray(raw_handmade,dtype=np.float32).reshape(-1,784)
    unscaled_mass=image_geometry(raw)["mass"]
    unscaled_topo=[topology(img) for img in raw.reshape(-1,28,28)]
    scale_rows=[]
    scale_geometry=[]
    for scale in SCALES:
        src=fixed_scale(raw,scale)
        dxdy=center_shifts(src,"mass")
        x=center_batch(src,"mass")
        geo=image_geometry(x)
        for i in range(len(y_hm)):
            if y_hm[i] in [LABELS.index(k) for k in COMPETITOR]:
                changed=int(topology(x[i].reshape(28,28))!=unscaled_topo[i])
            else:
                changed=-1
            scale_geometry.append({"seed":seed,"writer":int(writer_ids[i]),
              "index":int(i%640),"true":LABELS[int(y_hm[i])],
              "scale":scale,
              "bbox_width_after":float(geo["width"][i]),
              "bbox_height_after":float(geo["height"][i]),
              "bbox_longside_after":float(geo["long_side"][i]),
              "mass_ratio_after_over_original":float(geo["mass"][i]/
                                    max(unscaled_mass[i],1e-10)),
              "com_offset_after":float(geo["com_offset"][i]),
              "shift_x":int(dxdy[i,0]),"shift_y":int(dxdy[i,1]),
              "topology_changed_for_focus":changed})
        for trained,model in models.items():
            proba=np.asarray(model.predict_proba(x),dtype=float)
            labels=np.asarray(model.named_steps["model"].classes_,dtype=int)
            assert len(labels)==16 and np.array_equal(labels,np.arange(16))
            pred=proba.argmax(axis=1)
            for i in range(len(y_hm)):
                true=int(y_hm[i]); cl=LABELS[true]
                confuser=COMPETITOR.get(cl)
                true_score=proba[i,true]
                runner_up=max(proba[i,j] for j in range(16) if j!=true)
                rival_idx=LABELS.index(confuser) if confuser else -1
                scale_rows.append({
                    "seed":seed,"trained":trained,"writer":int(writer_ids[i]),
                    "index":int(i%640),"true":cl,
                    "scale":scale,"pred":LABELS[int(pred[i])],
                    "correct":int(pred[i]==true),
                    "p_true":float(true_score),
                    "p_best_other":float(runner_up),
                    "true_vs_best_margin":float(true_score-runner_up),
                    "confuser":confuser if confuser else "",
                    "p_confuser":float(proba[i,rival_idx]) if confuser else "",
                    "true_vs_pair_margin":float(true_score-proba[i,rival_idx]) if confuser else ""
                })
            group=evaluate_groups(y_hm,pred,writer_ids)
            print("EXP012_SCALE",seed,trained,scale,"all",
                  round(100*group["all"]["accuracy"],3),"digits",
                  round(100*group["digits"]["accuracy"],3),"letters",
                  round(100*group["letters"]["accuracy"],3),flush=True)
    save_csv(out/"scale_class_response.csv",scale_rows)
    save_csv(out/"scale_geometry.csv",scale_geometry)
    manifest={"experiment":"Normal-EXP011-EXP012","seed":seed,
        "archived_original_run":43785390739,
        "original_models_checked":["com","size","size_com"],
        "checkpoints_sha256":{m:historical[m]["model_sha256"] for m in TRAINED},
        "exact_original_prediction_counts_verified":True,
        "training_performed":False,
        "size_preprocessing":"EXP006 Combined 32 threshold 20px longside bilinear",
        "com_preprocessing":"EXP006 mass center threshold32 to 13.5px",
        "scales":list(SCALES),
        "scale_intervention":"affine around raw bbox center, then CoM centered",
        "handmade_n":2560,
        "official_validation_n":19200,
        "train_source_sha256":official["sha256"],
        "official_original_test_used":False,
        "professor_test_used":False,"handmade_training_used":False,
        "scale_topology_footprint_measured":True}
    (out/"manifest.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8")
    print("EXP011_EXP012_COMPLETE",seed,flush=True)

if __name__=="__main__":
    a=argparse.ArgumentParser()
    a.add_argument("--seed",type=int,choices=SEEDS,required=True)
    a.add_argument("--models-root",type=Path,required=True)
    a.add_argument("--out",type=Path,required=True)
    args=a.parse_args()
    run(args.seed,args.models_root,args.out)


## exp013_geometry_shortcut_binary

원본: [05_model_optimize/MLP/Normal/scripts/exp013_geometry_shortcut_binary.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp013_geometry_shortcut_binary.py)  
Git blob: `8808787653693e0b9cedeed19c6360cee5a86ef9`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

EXP013: Can geometry alone predict digit-vs-letter labels?
No original/professor Test. Fits ONLY logistic/tree binary probes, no MLP retraining.
Original EXP006/010 pixel processing and the existing official split are reused.

CLI 인자 (정적 추출): `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Normal/scripts/exp013_geometry_shortcut_binary.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""EXP013: Can geometry alone predict digit-vs-letter labels?
No original/professor Test. Fits ONLY logistic/tree binary probes, no MLP retraining.
Original EXP006/010 pixel processing and the existing official split are reused.
"""
from __future__ import annotations
import argparse, json, time
from pathlib import Path
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, accuracy_score, confusion_matrix
from exp006_size_com_comparison import ROOT, WRITERS, LABELS, load_official, load_writers, image_geometry, process, save_csv

METHODS=("raw","com","size","size_com")
FEATURES=("long_side","width","height","mass")
GROUPS={
 "size_only":("long_side",),
 "width_height":("width","height"),
 "ink_mass_only":("mass",),
 "all_four":FEATURES,
 "without_mass":("long_side","width","height"),
 "without_size":("width","height","mass"),
}
def extract(x,method,target,batch=1024):
    result=np.empty((len(x),len(FEATURES)),dtype=np.float64)
    for i in range(0,len(x),batch):
        v=np.asarray(x[i:i+batch],dtype=np.float32).reshape(-1,784)
        if method!="raw":
            v=process(v,method,target)
        geo=image_geometry(v)
        for k,name in enumerate(FEATURES):
            result[i:i+len(v),k]=geo[name]
    if not np.isfinite(result).all(): raise AssertionError("non-finite geometry")
    return result
def classifier(kind):
    if kind=="logreg":
        return make_pipeline(StandardScaler(),
            LogisticRegression(class_weight="balanced",max_iter=1500,random_state=42))
    if kind=="tree":
        return DecisionTreeClassifier(max_depth=3,min_samples_leaf=40,
                                       class_weight="balanced",random_state=42)
    raise ValueError(kind)
def metrics(y,prob,threshold=0.5):
    pred=(prob>=threshold).astype(int)
    mat=confusion_matrix(y,pred,labels=[0,1])
    return {"n":int(len(y)),"accuracy":float(accuracy_score(y,pred)),
      "balanced_accuracy":float(balanced_accuracy_score(y,pred)),
      "roc_auc":float(roc_auc_score(y,prob)),
      "digits_recall":float(mat[0,0]/mat[0].sum()),
      "letters_recall":float(mat[1,1]/mat[1].sum()),
      "digit_as_letter":int(mat[0,1]),"letter_as_digit":int(mat[1,0])}
def run(out):
    out=Path(out)
    if out.exists(): raise FileExistsError(out)
    out.mkdir(parents=True)
    official=load_official(ROOT/"02_data"/"datasets"/"MNIST_EMNIST_official_split_112000.npz")
    hm=load_writers(ROOT/"02_data"/"Hand_made")
    assert official["X_train"].shape==(76800,784) and official["X_val"].shape==(19200,784)
    X_h=np.concatenate([hm[w]["X"] for w in WRITERS])
    y_h=np.concatenate([hm[w]["y"] for w in WRITERS]).astype(int)
    w_h=np.repeat(WRITERS,640)
    assert len(y_h)==2560
    target=int(np.median(image_geometry(official["X_train"])["long_side"]))
    assert target==20
    y_tr=(np.asarray(official["y_train"])>=10).astype(int)
    y_va=(np.asarray(official["y_val"])>=10).astype(int)
    y_h=(y_h>=10).astype(int)
    rows=[];dists=[];examples=[]
    start=time.monotonic()
    for method in METHODS:
        train=extract(official["X_train"],method,target)
        valid=extract(official["X_val"],method,target)
        hand=extract(X_h,method,target)
        for k,f in enumerate(FEATURES):
            for family in (0,1):
                v=train[y_tr==family,k]
                dists.append({"preprocessing":method,"feature":f,
                  "family":"digits" if family==0 else "letters",
                  "n":len(v),"mean":float(np.mean(v)),"median":float(np.median(v)),
                  "p10":float(np.quantile(v,0.1)),"p90":float(np.quantile(v,0.9))})
        for name,features in GROUPS.items():
            idx=[FEATURES.index(f) for f in features]
            for kind in ("logreg","tree"):
                probe=classifier(kind)
                probe.fit(train[:,idx],y_tr)
                for dataset,X,y,w in (("OfficialVal",valid,y_va,None),
                                      ("Handmade",hand,y_h,w_h)):
                    prob=np.asarray(probe.predict_proba(X[:,idx])[:,1])
                    base={"preprocessing":method,"features":name,
                          "classifier":kind,"dataset":dataset}
                    rows.append({**base,"group":"all",**metrics(y,prob)})
                    if w is not None:
                        for writer in WRITERS:
                            m=w==writer
                            rows.append({**base,"group":f"writer_{writer}",
                                         **metrics(y[m],prob[m])})
                if name=="all_four" and kind=="logreg":
                    coefficients=probe.named_steps["logisticregression"].coef_.ravel()
                    for f,coef in zip(features,coefficients):
                        examples.append({"method":method,"feature":f,
                          "standardized_logistic_coef":float(coef)})
        print("EXP013",method,"processed",len(train),len(valid),len(hand),
              "seconds",round(time.monotonic()-start,1),flush=True)
    save_csv(out/"binary_probe_metrics.csv",rows)
    save_csv(out/"class_family_feature_distributions.csv",dists)
    save_csv(out/"standardized_coefficients.csv",examples)
    manifest={"experiment":"EXP013","target_longside_px":target,"foreground_threshold":32,
      "feature_names":list(FEATURES),"feature_sets":{k:list(v) for k,v in GROUPS.items()},
      "binary_targets":{"digits_0_9":0,"letters_A_F":1},"classifier_kinds":["logreg","tree"],
      "model_training":"binary probes only; original MLP unmodified",
      "n_official_train":76800,"n_official_val":19200,"n_handmade":2560,
      "official_sha256":official["sha256"],
      "writers_sha256":{str(w):hm[w]["sha256"] for w in WRITERS},
      "handmade_used_in_fit":False,"validation_used_in_fit":False,
      "original_test_used":False,"professor_test_used":False}
    (out/"manifest.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8")
    print("EXP013_COMPLETE",len(rows),len(dists),flush=True)
if __name__=="__main__":
    p=argparse.ArgumentParser()
    p.add_argument("--out",type=Path,required=True)
    run(p.parse_args().out)


## exp010_012_aggregate

원본: [05_model_optimize/MLP/Normal/scripts/exp010_012_aggregate.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp010_012_aggregate.py)  
Git blob: `4157bd9d506bbae337a260d3c98191d38c27ee98`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

Aggregate genuine, checked EXP010/011/012 results without inferring numbers.

CLI 인자 (정적 추출): `--input-root`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Normal/scripts/exp010_012_aggregate.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Aggregate genuine, checked EXP010/011/012 results without inferring numbers."""
from __future__ import annotations
import argparse
import csv
import json
from pathlib import Path
import numpy as np

SEEDS=(42,43,44)
METHODS=("com","size","size_com")
INPUTS=("raw","com","size","size_com")
FOCUS=("B","8","D","0","F","5")

def read_csv(path):
    with Path(path).open("r",encoding="utf-8-sig",newline="") as f:
        return list(csv.DictReader(f))
def save_csv(path,rows):
    if not rows:raise RuntimeError("No results for "+str(path))
    with Path(path).open("w",encoding="utf-8-sig",newline="") as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]))
        w.writeheader();w.writerows(rows)
def avg(vals):
    if not vals:raise AssertionError("Missing grouped comparison")
    return float(np.mean(vals))
def pct(x):return f"{x*100:.4f}%"
def ffloat(row,key):return float(row[key])
def subgroup(rows,**conditions):
    return [r for r in rows if all(str(r[k])==str(v) for k,v in conditions.items())]

def run(input_root,out):
    input_root,out=Path(input_root),Path(out)
    if out.exists():raise FileExistsError(out)
    out.mkdir(parents=True)
    geo=input_root/"exp010-geometry"
    if not geo.is_dir():raise FileNotFoundError(geo)
    geoman=json.loads((geo/"manifest.json").read_text(encoding="utf-8"))
    assert not geoman["official_original_test_used"]
    assert geoman["target_bbox_longside"]==20
    stat=read_csv(geo/"geometry_summary.csv")
    dists=read_csv(geo/"geometry_distance.csv")
    hgeo=read_csv(geo/"handmade_geometry_per_image.csv")
    cross=[];paired=[];errors=[];geometry=[];scale=[];sg=[];manifests={}
    for seed in SEEDS:
        d=input_root/f"exp011-012-seed{seed}"
        if not d.is_dir():raise FileNotFoundError(d)
        manifest=json.loads((d/"manifest.json").read_text(encoding="utf-8"))
        assert manifest["seed"]==seed
        assert manifest["exact_original_prediction_counts_verified"]
        assert not manifest["training_performed"]
        assert not manifest["official_original_test_used"]
        assert not manifest["professor_test_used"]
        assert not manifest["handmade_training_used"]
        assert manifest["train_source_sha256"]==geoman["official_sha256"]
        manifests[str(seed)]=manifest
        cross+=read_csv(d/"cross_metrics.csv")
        paired+=read_csv(d/"paired_effects.csv")
        errors+=read_csv(d/"pair_errors.csv")
        geometry+=read_csv(d/"size_com_geometry_and_outcomes.csv")
        scale+=read_csv(d/"scale_class_response.csv")
        sg+=read_csv(d/"scale_geometry.csv")
    save_csv(out/"all_cross_metrics.csv",cross)
    save_csv(out/"all_paired_effects.csv",paired)
    save_csv(out/"all_pair_errors.csv",errors)
    save_csv(out/"all_size_center_outcomes.csv",geometry)
    save_csv(out/"all_scale_response.csv",scale)
    save_csv(out/"all_scale_geometry.csv",sg)
    summary=[]
    for method in METHODS:
        for dataset in ("OfficialVal","Handmade"):
            for inp in INPUTS:
                for group in ("all","digits","letters",*[f"class_{c}" for c in "0123456789ABCDEF"]):
                    xs=subgroup(cross,trained=method,dataset=dataset,input=inp,group=group)
                    if len(xs)!=3:raise AssertionError((method,dataset,inp,group,len(xs)))
                    summary.append({"trained":method,"dataset":dataset,"input":inp,"group":group,
                        "n_per_seed":int(xs[0]["n"]),
                        "mean_accuracy":avg([ffloat(x,"accuracy") for x in xs]),
                        "seed_42_acc":float(xs[0]["accuracy"]),
                        "seed_43_acc":float(xs[1]["accuracy"]),
                        "seed_44_acc":float(xs[2]["accuracy"])})
    save_csv(out/"exp011_accuracy_matrix.csv",summary)
    comparisons=[]
    for contrast in sorted(set(r["contrast"] for r in paired)):
        for group in ("all","digits","letters",*[f"class_{c}" for c in "0123456789ABCDEF"]):
            xs=subgroup(paired,contrast=contrast,group=group)
            assert len(xs)==3,(contrast,group,len(xs))
            comparisons.append({"contrast":contrast,"group":group,
                "n":int(xs[0]["n"]),"mean_fixed":avg([ffloat(x,"fixed") for x in xs]),
                "mean_new_wrong":avg([ffloat(x,"new_wrong") for x in xs]),
                "mean_net_correct":avg([ffloat(x,"net_correct") for x in xs]),
                "mean_delta_accuracy_pp":avg([ffloat(x,"delta_accuracy_pp") for x in xs])})
    save_csv(out/"exp011_paired_summary.csv",comparisons)

    # Size-only diagonal -> Size+CoM diagonal, class/writer geometric shift
    diagnostic=[]
    for group,sub in [
        ("all",geometry),
        ("digits",[r for r in geometry if r["class"].isdigit()]),
        ("letters",[r for r in geometry if r["class"] in "ABCDEF"]),
        ("recovered",[r for r in geometry if r["size_model_size_correct"]=="0"
             and r["size_com_model_size_com_correct"]=="1"]),
        ("newly_wrong",[r for r in geometry if r["size_model_size_correct"]=="1"
             and r["size_com_model_size_com_correct"]=="0"])
    ]:
        for cl in ("all",*[f"class_{c}" for c in "0123456789ABCDEF"]):
            sample=sub if cl=="all" else [r for r in sub if r["class"]==cl[6:]]
            if not sample:continue
            diagnostic.append({"status":group,"class_or_all":cl,"n_across_seeds":len(sample),
                "mean_com_distance_before":avg([ffloat(x,"com_distance_before") for x in sample]),
                "mean_com_distance_after":avg([ffloat(x,"com_distance_after") for x in sample]),
                "mean_shift_distance":avg([ffloat(x,"size_com_shift_distance") for x in sample]),
                "mean_mass_ratio_after":avg([ffloat(x,"mass_after")/
                    max(ffloat(x,"mass_before"),1e-12) for x in sample]),
                "fraction_nonzero_shift":avg([ffloat(x,"size_com_shift_distance")>0 for x in sample])})
    save_csv(out/"exp011_center_geometry_by_outcome.csv",diagnostic)

    # EXP010 reference geometry and domain differences
    g010=[]
    for cls in ("all","digits","letters",*[f"class_{c}" for c in "0123456789ABCDEF"]):
        for method in ("raw","size","size_com"):
            for feature in ("width","height","long_side","com_offset","mass"):
                a=subgroup(stat,dataset="OfficialTrain",preprocessing=method,
                           group=cls,feature=feature)
                b=subgroup(stat,dataset="Handmade",preprocessing=method,
                           group=cls,feature=feature)
                dx=subgroup(dists,preprocessing=method,segment=cls,
                            feature=feature)
                if not a or not b:raise AssertionError((cls,method,feature))
                g010.append({"group":cls,"preprocessing":method,
                    "feature":feature,"train_median":float(a[0]["median"]),
                    "handmade_median":float(b[0]["median"]),
                    "train_mean":float(a[0]["mean"]),
                    "handmade_mean":float(b[0]["mean"]),
                    "train_handmade_w1":float(dx[0]["train_vs_handmade_w1"]) if dx else ""})
    save_csv(out/"exp010_geometry_core_summary.csv",g010)

    # EXP012 pair margins (pre-registered 6 focus directions), 3 seeds, 3 MLPs
    pairs=[]
    all_classes=[]
    for trained in METHODS:
        for cl in FOCUS:
            for factor in (.8,.9,1.0,1.1,1.2):
                xs=subgroup(scale,trained=trained,true=cl,scale=factor)
                assert len(xs)==480,(trained,cl,factor,len(xs)) # 160/class ×3 seeds
                pairs.append({"trained":trained,"class":cl,
                    "comparison":cl+" vs "+xs[0]["confuser"],
                    "scale":factor,"n_across_seeds":len(xs),
                    "true_recall":avg([int(x["correct"]) for x in xs]),
                    "mean_pair_margin":avg([float(x["true_vs_pair_margin"]) for x in xs]),
                    "mean_p_true":avg([float(x["p_true"]) for x in xs]),
                    "mean_p_confuser":avg([float(x["p_confuser"]) for x in xs]),
                    "n_pred_as_confuser":sum(int(x["pred"]==x["confuser"]) for x in xs)})
        for group in ("digits","letters"):
            for factor in (.8,.9,1.0,1.1,1.2):
                rows=[r for r in scale if r["trained"]==trained and
                    float(r["scale"])==factor and
                    (r["true"] in "0123456789" if group=="digits" else
                     r["true"] in "ABCDEF")]
                assert len(rows)==(1600 if group=="digits" else 960)*3
                all_classes.append({"trained":trained,"group":group,
                    "scale":factor,"n_across_seeds":len(rows),
                    "accuracy":avg([int(r["correct"]) for r in rows])})
    save_csv(out/"exp012_scale_pair_margins.csv",pairs)
    save_csv(out/"exp012_scale_group_accuracy.csv",all_classes)
    topo=[]
    for cl in FOCUS:
        for factor in (.8,.9,1.0,1.1,1.2):
            sample=subgroup(sg,true=cl,scale=factor)
            assert len(sample)==480,(cl,factor,len(sample))
            topo.append({"class":cl,"scale":factor,"n_across_seeds":len(sample),
                "fraction_topology_changed":avg([float(x["topology_changed_for_focus"]) for x in sample]),
                "median_bbox_longside":float(np.median([float(x["bbox_longside_after"]) for x in sample])),
                "mean_ink_mass_ratio":avg([float(x["mass_ratio_after_over_original"]) for x in sample]),
                "mean_com_offset":avg([float(x["com_offset_after"]) for x in sample])})
    save_csv(out/"exp012_geometry_confound_checks.csv",topo)

    # Precompute class mean accuracy at crossover, compare effect vs historic.
    lookup={(r["trained"],r["dataset"],r["input"],r["group"]):float(r["mean_accuracy"])
            for r in summary}
    def acc(method,inp,group,dataset="Handmade"):
        return lookup[method,dataset,inp,group]
    line=["# Normal EXP유010·011·012 — 실제 측정 종합 결과","",
          "**상태: 3가설 검증 완료. EXP유006 원본 9개 체크포인트 해시·예측 정답 수 재검증, 재학습 없음.**",
          "","- 공식 Train 76,800 · Validation 19,200 · Hand-made 2,560 (작성자 41-44).",
          "- EXP유006과 완전히 같은 Combined Size(threshold32, longside20, bilinear), CoM(threshold32, 정수 이동).",
          "- 공식 Original Test, 교수님 Test, 새로운 Hand-made 학습 없음.",
          "- 모든 수치는 동일한 3 Seed(42·43·44)의 성공한 기존 MLP를 사용한 실측이다.",
          "",
          "## EXP유011 원본 모델 vs 평가 입력 교차 비교",
          "",
          "| 학습 모델 | 평가 입력 | Hand-made 숫자 | Hand-made 문자 | 공식 Validation |",
          "|---|---|---:|---:|---:|"]
    for model in METHODS:
        for inp in ("com","size","size_com"):
            line.append(f"| {model} | {inp} | {pct(acc(model,inp,'digits'))} | "
                  f"{pct(acc(model,inp,'letters'))} | "
                  f"{pct(acc(model,inp,'all','OfficialVal'))} |")
    line+=["","## EXP유010 주요 크기 분포","",
           "| 문자/숫자 | 전처리 | 공식 Training BBox 긴 변 중앙값 | Hand-made BBox 긴 변 중앙값 | W1 거리 |",
           "|---|---|---:|---:|---:|"]
    for cl in ("B","8","D","0","F","5"):
        for method in ("raw","size","size_com"):
            r=[x for x in g010 if x["group"]=="class_"+cl and x["preprocessing"]==method
               and x["feature"]=="long_side"][0]
            w=r["train_handmade_w1"]
            line.append(f"| {cl} | {method} | {r['train_median']:.2f}px | "
                        f"{r['handmade_median']:.2f}px | {w:.3f} |")
    line+=["","## EXP유011 중요 결과","",
      "- Size-only vs Size→CoM 대각선 정확도 차이는 **학습도 입력도 동시에 변경**한 값이다.",
      "- 같은 학습 모델의 Size 입력과 Size→CoM 입력 차이가 '입력 위치 효과'를 직접 점검하는 비교다.",
      "- 반대로 동일한 입력에 Size-only 모델과 Size→CoM 모델을 적용한 차이는 모델 학습의 영향이다.",
      "- 서로 다른 train/validation 이미지 분포를 교차 적용하는 경우에는 OOD 효과가 섞일 수 있다.",
      "",
      "## EXP유012 크기 개입 — 중심 정렬 유지","",
      "| 모델 | 실제 클래스 | 0.8x pair margin | 1.0x pair margin | 1.2x pair margin |",
      "|---|---|---:|---:|---:|"]
    for model in ("com","size_com"):
        for cl in FOCUS:
            rr=[r for r in pairs if r["trained"]==model and r["class"]==cl]
            mar={r["scale"]:r["mean_pair_margin"] for r in rr}
            line.append(f"| {model} | {cl} | {mar[.8]:+.4f} | "
                        f"{mar[1.0]:+.4f} | {mar[1.2]:+.4f} |")
    line+=["","- Pair margin = P(true class) − P(confusable class); positive favors true.",
           "- 크기 변화는 획 굵기, 보간, 일부 픽셀 손상까지 바꿀 수 있으므로 단일한 '크기 단서'의 인과 증거로 과도하게 해석하지 않는다.",
           "- 형상 위상 변화율·잉크 질량 변화·CoM 잔여 오프셋은 exp012_geometry_confound_checks.csv에 보존한다.",
           "- 4명 작성자는 반복 사용한 개발 데이터이며 독립 Test 성능을 의미하지 않는다.",
           "",
           "## 파일 안내","",
           "- exp010_geometry_core_summary.csv: 원본·Size·SizeCoM 분포의 실제 중앙값과 W1.",
           "- exp011_accuracy_matrix.csv: Train Model × Eval Input, 3-seed 각 정확도.",
           "- exp011_paired_summary.csv: 같은 이미지의 corrected/new_wrong, 고정 모델 입력 변화.",
           "- exp011_center_geometry_by_outcome.csv: CoM 복구량과 오답 회복/악화의 관계.",
           "- exp012_scale_pair_margins.csv: 클래스쌍 예측 반응 및 올바른 클래스 Margin.",
           "- exp012_geometry_confound_checks.csv: 위상 변화, 획 질량, CoM 통제 검증.",
           "- all_*: 전수 예측 및 원시 계측 결과(Artifact).",
           "",
           "**최종 모델·main 변경 없음.**"]
    (out/"SUMMARY.md").write_text("\n".join(line)+"\n",encoding="utf-8")
    (out/"run_manifest.json").write_text(json.dumps({
        "experiments":["EXP010","EXP011","EXP012"],
        "models":3,"seeds":list(SEEDS),"models_per_seed":list(METHODS),
        "model_run":43785390739,
        "checkpoint_hash_and_baseline_counts_verified":True,
        "test_used":False,"training_used":False,
        "all_class_geometry_n":len(stat),"scale_response_rows":len(scale),
        "source_sha256":geoman["official_sha256"],
    },indent=2,ensure_ascii=False),encoding="utf-8")
    print("EXP010_011_012_AGGREGATION_COMPLETE",flush=True)
    print("\n".join(line),flush=True)

if __name__=="__main__":
    p=argparse.ArgumentParser()
    p.add_argument("--input-root",type=Path,required=True)
    p.add_argument("--out",type=Path,required=True)
    a=p.parse_args()
    run(a.input_root,a.out)


## exp013_014_aggregate

원본: [05_model_optimize/MLP/Normal/scripts/exp013_014_aggregate.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp013_014_aggregate.py)  
Git blob: `5fa22a60cd93a8626323f1228395c4095a47129b`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

EXP013/014: aggregate independent three-seed measurements and family-probability robustness.

CLI 인자 (정적 추출): `--inputs`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Normal/scripts/exp013_014_aggregate.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""EXP013/014: aggregate independent three-seed measurements and family-probability robustness."""
from __future__ import annotations
import argparse,csv,json,statistics
from pathlib import Path
def read(path):
    with Path(path).open("r",encoding="utf-8-sig",newline="") as f:
        return list(csv.DictReader(f))
def save(path,rows):
    if not rows:raise RuntimeError(f"No rows for {path}")
    with Path(path).open("w",encoding="utf-8-sig",newline="") as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]))
        w.writeheader();w.writerows(rows)
def avg(nums):
    return sum(nums)/len(nums)
def run(source,out):
    source=Path(source);out=Path(out)
    if out.exists():raise FileExistsError(out)
    out.mkdir(parents=True)
    exp013=next(source.glob("exp013-binary*/binary_probe_metrics.csv"))
    binary=read(exp013)
    binary_final=[r for r in binary if r["group"]=="all"]
    save(out/"exp013_binary_probe_scores.csv",binary_final)
    root=source
    exps=[root/f"exp014-seed{s}" for s in (42,43,44)]
    for p in exps:
        assert (p/"accuracy_metrics.csv").is_file(),p
        assert (p/"scale_image_family_probabilities.csv").is_file(),p
        m=json.loads((p/"manifest.json").read_text(encoding="utf-8"))
        assert m["original_checkpoint_hash_and_predictions_verified"]
        assert not m["original_test_used"] and not m["professor_test_used"]
        assert not m["handmade_used_in_fit"] and not m["validation_used_in_fit"]
    all_m=[row for p in exps for row in read(p/"accuracy_metrics.csv")]
    save(out/"exp014_all_seed_metrics.csv",all_m)
    by={}
    for r in all_m:
        if r["group"] not in ("all","digits","letters"):continue
        key=(r["model"],r["dataset"],r["group"])
        by.setdefault(key,[]).append(float(r["accuracy"]))
    summary=[]
    for (model,dataset,group),values in sorted(by.items()):
        assert len(values)==3,(model,dataset,group,len(values))
        summary.append({"model":model,"dataset":dataset,"group":group,
            "n_seeds":3,"mean_accuracy":avg(values),
            "std_accuracy":statistics.stdev(values),
            "seed42":values[0],"seed43":values[1],"seed44":values[2]})
    save(out/"exp014_accuracy_summary.csv",summary)
    # The same source image and model seed are paired across five scaling interventions.
    per_seed={}
    for p in exps:
        records=read(p/"scale_image_family_probabilities.csv")
        idx={}
        for r in records:
            key=(int(r["seed"]),r["model"],int(r["writer"]),int(r["index"]))
            idx.setdefault(key,{})[float(r["scale"])]=r
        for key,v in idx.items():
            assert set(v)=={.8,.9,1.,1.1,1.2},key
            seed,model,writer,i=key
            truth=v[1.]["true"]
            family="digits" if truth in "0123456789" else "letters"
            for scale in (.8,.9,1.1,1.2):
                p0=float(v[1.]["p_letter"]);ps=float(v[scale]["p_letter"])
                m0=p0>=.5;ms=ps>=.5
                block=(seed,model,family,scale)
                s=per_seed.setdefault(block,{"n":0,"delta_sum":0.,"absolute_sum":0.,"flips":0})
                s["n"]+=1;s["delta_sum"]+=ps-p0
                s["absolute_sum"]+=abs(ps-p0);s["flips"]+=int(ms!=m0)
    sensitivity=[]
    for (seed,model,family,scale),v in sorted(per_seed.items()):
        sensitivity.append({"seed":seed,"model":model,"family":family,"scale":scale,
           "n_images":v["n"],"mean_signed_delta_p_letter":v["delta_sum"]/v["n"],
           "mean_abs_delta_p_letter":v["absolute_sum"]/v["n"],
           "family_prediction_flip_rate":v["flips"]/v["n"]})
    save(out/"exp014_family_scale_sensitivity_by_seed.csv",sensitivity)
    grouped={}
    for r in sensitivity:
        key=(r["model"],r["family"],r["scale"])
        grouped.setdefault(key,[]).append(r)
    agg=[]
    for (model,family,scale),v in sorted(grouped.items()):
        assert len(v)==3
        agg.append({"model":model,"family":family,"scale":scale,
          "n_seeds":3,"n_unique_images_per_seed":v[0]["n_images"],
          "mean_signed_delta_p_letter":avg([q["mean_signed_delta_p_letter"] for q in v]),
          "mean_abs_delta_p_letter":avg([q["mean_abs_delta_p_letter"] for q in v]),
          "mean_family_flip_rate":avg([q["family_prediction_flip_rate"] for q in v])})
    save(out/"exp014_family_scale_sensitivity_summary.csv",agg)
    md=["# EXP013·014 — Actual measured results (3 seeds)","",
      "Source is GitHub Actions for this experiment branch. No original/professor Test.",
      "EXP013 binary-probe models were trained on Official Train ONLY.",
      "EXP014 MLP variants were trained on Official Train 76,800 samples each.",
      "Hand-made is a repeatedly used development set, not an independent final test.",
      "",
      "## EXP013 geometry-only binary classification (Official Validation)",
      "",
      "| Features after | Feature set | Probe | Balanced Acc | ROC AUC |",
      "|---|---|---|---:|---:|"]
    for r in binary_final:
        if r["dataset"]=="OfficialVal" and r["features"] in ("all_four","size_only","ink_mass_only") and r["classifier"]=="logreg":
            md.append(f"| {r['preprocessing']} | {r['features']} | {r['classifier']} | "
                      f"{100*float(r['balanced_accuracy']):.3f}% | {float(r['roc_auc']):.4f} |")
    md+=["","## EXP014 Official Validation and Handmade (three-seed means)","",
         "| Model | Official Val | Handmade digits | Handmade letters |",
         "|---|---:|---:|---:|"]
    names=sorted({r["model"] for r in summary})
    lookup={(r["model"],r["dataset"],r["group"]):r["mean_accuracy"] for r in summary}
    for name in names:
        md.append(f"| {name} | {100*lookup[name,'OfficialVal','all']:.3f}% | "
                  f"{100*lookup[name,'Handmade','digits']:.3f}% | "
                  f"{100*lookup[name,'Handmade','letters']:.3f}% |")
    md+=["","## EXP014 Size response of full preprocessing pipelines",
         "Mean absolute change in P(letter) vs 1.0x; smaller indicates more robust behavior.",
         "Also see per-seed family flip rate; robustness alone is not accuracy.",
         "",
         "| Model | Class family | Scale | Mean |Delta P(letter)| | Flip rate |",
         "|---|---|---:|---:|---:|"]
    for r in agg:
        if r["scale"] in (.8,1.2):
            md.append(f"| {r['model']} | {r['family']} | {r['scale']:.1f} | "
                      f"{r['mean_abs_delta_p_letter']:.4f} | "
                      f"{100*r['mean_family_flip_rate']:.2f}% |")
    md+=["","## Interpretation boundaries",
      "- EXP013: geometry-only binary accuracy is evidence a shortcut feature exists in the dataset; it is not by itself proof that the MLP uses it.",
      "- EXP014: compare augmentation against frozen original EXP006 CoM and SizeCoM controls at the same preprocessing and original developer data.",
      "- CoM augmentation models and SizeCoM augmentation model should be compared to the correspondingly preprocessed archived controls.",
      "- Scale perturbations alter thickness, mass, and pixels; improvements do not prove a unique isolated feature caused the change.",
      "- Original Test/professor Test were never used. Do not adopt new pipeline without approval.",
      ""]
    (out/"SUMMARY.md").write_text("\n".join(md),encoding="utf-8")
    (out/"manifest.json").write_text(json.dumps({"source_seeds":[42,43,44],
      "source_exp013":str(exp013.relative_to(source)),"independent_test_used":False,
      "main_pipeline_changed":False},indent=2),encoding="utf-8")
    print("EXP013_EXP014_REVIEW_COMPLETE",len(binary_final),len(summary),len(agg),flush=True)
if __name__=="__main__":
    p=argparse.ArgumentParser();p.add_argument("--inputs",type=Path,required=True);p.add_argument("--out",type=Path,required=True)
    a=p.parse_args();run(a.inputs,a.out)


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")